# MathAgent Showcase

选择 **Run → Run All Cells**。提交单元格通过本机 Showcase HTTP API 创建会话并提交题目，随后显示执行事件和答复。可修改 `problem` 后重新运行。

默认 `live` 模式会调用 Intern。请先在仓库的 `.env` 或环境变量中配置 `INTERN_API_KEY`。用 `--mode fake` 启动时使用本地模拟结果。

In [ ]:
import os
import sys
from pathlib import Path
from threading import Thread
from time import sleep
from uuid import uuid4

import httpx
from werkzeug.serving import make_server

cwd = Path.cwd()
repo = next((path for path in (cwd, *cwd.parents) if (path / "showcase_server.py").is_file()), None)
if repo is None:
    raise RuntimeError("请从仓库运行 scripts/start_showcase.py notebook。")
sys.path.insert(0, str(repo))

from showcase.flask_app import create_app

mode = os.environ.get("MATHAGENT_SHOWCASE_MODE", "live")
print("Showcase 模式", mode)

In [ ]:
problem = "求解 x² − 1 = 0。"
data_dir = repo / "artifacts" / "showcase-notebook" / f"job-{uuid4().hex}"
app = create_app({"SHOWCASE_MODE": mode, "SHOWCASE_DATA_DIR": str(data_dir)})

with make_server("127.0.0.1", 0, app, threaded=True) as server:
    worker = Thread(target=server.serve_forever, daemon=True)
    worker.start()
    try:
        with httpx.Client(base_url=f"http://127.0.0.1:{server.server_port}", trust_env=False) as client:
            print("健康状态", client.get("/health").raise_for_status().json()["status"])
            auth = client.get("/api/check_auth").raise_for_status().json()
            headers = {"X-CSRF-Token": auth["csrf_token"]}
            session = client.post("/api/new_chat", headers=headers).raise_for_status().json()
            job = client.post(
                "/api/agent/jobs",
                json={"session_id": session["new_session_id"], "message": problem},
                headers={**headers, "Idempotency-Key": str(uuid4())},
            ).raise_for_status().json()
            job_path = f"/api/agent/jobs/{job['job_id']}"
            print("任务 ID", job["job_id"])

            while job["status"] in {"queued", "running"}:
                sleep(1)
                job = client.get(job_path).raise_for_status().json()
            events = client.get(f"{job_path}/events?snapshot=1").raise_for_status().json()["events"]
            answer = client.get(f"{job_path}/answer").raise_for_status().json()
    finally:
        server.shutdown()
        worker.join()
        app.extensions["showcase_workspace_service"].close()
        app.extensions["showcase_run_service"].close()
        app.extensions["loopback_store"].close()

print("执行事件", " → ".join(event["event"] for event in events))
print("任务状态", answer["status"])
print("执行模式", mode)
answer